# Fine-tuning Qwen3.5-4B di Google Colab

Panduan lengkap langkah demi langkah ada di
[`docs/PANDUAN_TRAINING.md`](https://github.com/PvzKing/Dataset/blob/main/docs/PANDUAN_TRAINING.md).

**Alur singkat:**
1. Pilih GPU di menu *Runtime → Change runtime type*. T4 tersedia gratis.
2. Jalankan sel **1–4** (persiapan).
3. Pilih salah satu cara:
   - **Mode A:** jalankan sendiri dengan satu sel.
   - **Mode B:** biarkan **Claude** yang menjalankan, memantau, dan memperbaiki masalah lewat terminal.

Semua hasil (checkpoint, adapter, contoh website) disimpan ke Google Drive. Jika sesi putus, jalankan ulang sel 1–4,
lalu mode yang sama, dan training **dilanjutkan dari checkpoint terakhir**.

## 1. Cek GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Instal Unsloth
Butuh ±2–4 menit. Jika Colab meminta *Restart session*, klik tombolnya lalu lanjut ke sel 3. Sel ini tidak perlu diulang.

In [ ]:
%%capture
!pip install --upgrade unsloth

## 3. Konfigurasi
Pengaturan bawaan sudah sesuai rekomendasi: Qwen3.5-4B dengan data `own`.

In [ ]:
MODEL       = "qwen3.5-4b"      # "qwen3.5-4b" atau "qwen2.5-coder-7b" (pembanding)
DATASET     = "own"             # "own" = hanya dataset ini (disarankan), "mix" = ditambah dataset umum
MAX_SEQ_LEN = 8192              # turunkan ke 6144 jika muncul "CUDA out of memory"
DRIVE_ROOT  = "/content/drive/MyDrive/finetune-id"
REPO_URL    = "https://github.com/PvzKing/Dataset.git"
BRANCH      = "main"

## 4. Hubungkan Google Drive dan ambil repo
Colab akan meminta izin akses Google Drive. Pilih akun Google Anda, lalu izinkan.

Jika repo **privat**:
1. Buat *fine-grained token* GitHub dengan akses baca ke repo ini.
2. Simpan token di panel 🔑 *Secrets* (kiri) dengan nama `GITHUB_TOKEN`.
3. Aktifkan *Notebook access* untuk secret itu.

In [ ]:
import json
import os
import subprocess
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)


def sh(*cmd):
    if subprocess.run(cmd).returncode != 0:
        raise RuntimeError(f"perintah gagal: {cmd[0]} {cmd[1] if len(cmd) > 1 else ''}")


if not os.path.isdir("/content/Dataset"):
    clone_url = REPO_URL
    try:
        from google.colab import userdata
        clone_url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # tanpa token: repo publik
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, "/content/Dataset")
    sh("git", "-C", "/content/Dataset", "remote", "set-url", "origin", REPO_URL)  # token tidak disimpan di .git/config
os.chdir("/content/Dataset")

RUN_DIR = f"{DRIVE_ROOT}/{MODEL}-{DATASET}"
ARGS = f"--model {MODEL} --dataset {DATASET} --out {DRIVE_ROOT} --max-seq-len {MAX_SEQ_LEN}"
with open("run_config.json", "w") as f:  # dibaca Claude di mode B
    json.dump({"model": MODEL, "dataset": DATASET, "out": DRIVE_ROOT, "max_seq_len": MAX_SEQ_LEN}, f)
sh("git", "log", "-1", "--format=repo siap, commit %h  %s")
print("hasil akan disimpan di", RUN_DIR)

---
## Mode A: jalankan sendiri

Satu sel ini menjalankan seluruh proses:
1. cek data;
2. training;
3. simpan adapter;
4. buat dua website uji.

Progress dan ETA tampil di bawah sel. Biarkan tab browser tetap terbuka.

In [ ]:
!python scripts/train_unsloth.py train {ARGS}

### Lihat hasil

In [ ]:
import html
from IPython.display import HTML, display

status = json.load(open(f"{RUN_DIR}/status.json"))
print(json.dumps({k: status.get(k) for k in ("stage", "message", "result", "tests", "hint")}, indent=1, ensure_ascii=False))
for name in sorted(os.listdir(RUN_DIR)):
    if name.endswith(".html"):
        page = open(f"{RUN_DIR}/{name}", encoding="utf-8").read()
        display(HTML(f'<h3>{name}</h3><iframe srcdoc="{html.escape(page)}" '
                     'style="width:100%;height:600px;border:1px solid #ccc"></iframe>'))

---
## Mode B: Claude yang mengontrol

Claude Code berjalan di terminal VM Colab ini. Claude bisa menjalankan training, membaca log, memantau GPU,
menangani error yang umum (kehabisan memori, paket bermasalah, proses mati), lalu melaporkan hasilnya ke Anda.
Aturan kerjanya ada di [`docs/RUNBOOK_CLAUDE.md`](https://github.com/PvzKing/Dataset/blob/main/docs/RUNBOOK_CLAUDE.md).

**Syarat:** akun Claude Pro, Max, Team, atau Enterprise (atau akun API Console).

Jalankan sel di bawah, lalu ikuti langkah di bawahnya.

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash
!pip install -q colab-xterm
%load_ext colabxterm
%xterm

Di jendela terminal yang muncul:

1. Ketik perintah ini lalu Enter:
   ```
   cd /content/Dataset && export PATH="$HOME/.local/bin:$PATH" && claude
   ```
2. **Login:**
   - Pilih *Claude account*. Buka link yang muncul di tab baru, login, lalu salin kode yang diberikan.
   - Tempel kode itu di terminal (Ctrl+Shift+V atau klik kanan → Paste).
   - Jika muncul pertanyaan *trust this folder*, pilih **Yes**.
3. Ketik perintah untuk Claude, misalnya:
   ```
   Latih model sesuai run_config.json mengikuti docs/RUNBOOK_CLAUDE.md, pantau sampai selesai, lalu laporkan hasilnya.
   ```
4. **Izin:** saat Claude meminta izin menjalankan perintah, pilih **Yes, and don't ask again** untuk perintah yang aman
   seperti `python scripts/train_unsloth.py`, `tail`, `cat`, `nvidia-smi`, dan `sleep`. Dengan begitu Claude bisa
   memantau tanpa terus bertanya.
5. Biarkan tab Colab tetap terbuka. Anda bisa bertanya kapan saja di terminal, misalnya "gimana progresnya?".

**Mengontrol dari HP (Remote Control):** setelah login dan *trust* di langkah 2, keluar dengan `/exit`, lalu jalankan
`claude remote-control` dan buka sesinya di aplikasi Claude (tab *Code*) atau claude.ai/code. Menurut FAQ Colab, *remote control* dan bekerja di luar UI notebook
termasuk aktivitas yang dibatasi di **tier gratis**, sehingga runtime bisa dihentikan. Pakai cara ini hanya jika
saldo *compute unit* Colab Anda positif (Colab Pro atau Pay As You Go).

---
## Opsional: ekspor GGUF untuk Ollama / llama.cpp
Jalankan setelah training selesai. Butuh ±15–40 menit dan ±3 GB ruang di Drive (untuk 4B).

In [ ]:
!python scripts/train_unsloth.py gguf {ARGS}